# UAV–LiDAR biomass estimation with mapped field trees (R)

This self-contained notebook adapts Fu et al. (2025), *Comparison of UAV-LiDAR-driven biomass estimation approaches in planted forests with different management*, to the Cal Poly arboretum inventory.

It uses the mapped trees for tree-centric calibration, keeps height-only trees available for segmentation and height validation, and excludes trees without defensible biomass from biomass-model fitting. It does **not** silently interpret incomplete footprint sums as complete reference biomass.

Paper: https://doi.org/10.1080/17538947.2025.2576910

### Expected inputs

- UAV point cloud: `.las` or `.laz`
- normalized canopy-height model: `.tif` or `.tiff`
- six 12.5 m-diameter footprint polygons: `.gpkg`, `.shp`, or `.geojson`
- the Excel inventory containing `Footprint Trees` and `Biomass Calcs`
- optional overall arboretum boundary for wall-to-wall prediction

Run cells from top to bottom. The input cell can discover files in one project folder or prompt for each file separately.


## 1. Packages

Install once if needed:

```r
install.packages(c("lidR", "sf", "terra", "dplyr", "purrr", "tidyr", "tibble", "ggplot2", "readxl", "readr", "clue"))
IRkernel::installspec()
```


In [ ]:

required_packages <- c("lidR", "sf", "terra", "dplyr", "purrr", "tidyr", "tibble", "ggplot2", "readxl", "readr", "clue")
missing_packages <- setdiff(required_packages, rownames(installed.packages()))
if (length(missing_packages)) stop("Install these packages first: ", paste(missing_packages, collapse = ", "))
suppressPackageStartupMessages({
  library(lidR); library(sf); library(terra); library(dplyr); library(purrr)
  library(tidyr); library(tibble); library(ggplot2); library(readxl); library(readr); library(clue)
  library(tidyterra)
})
set.seed(42)


## 2. Select files

Select the necessary set-up files 

In [ ]:
# ============================================================
# Input selection and analysis settings
# ============================================================

ask <- function(message, default = NULL) {
  prompt <- if (is.null(default)) paste0(message, ": ")
            else paste0(message, " [", default, "]: ")
  answer <- trimws(readline(prompt))
  if (!nzchar(answer) && !is.null(default)) default else answer
}

pick_file <- function(label, exts) {
  filters <- matrix(
    c(paste0(label, " (", paste0("*.", exts, collapse = ";"), ")"),
      paste0("*.", exts, collapse = ";"),
      "All files (*.*)", "*.*"),
    ncol = 2, byrow = TRUE
  )
  path <- utils::choose.files(caption = paste("Select:", label),
                               multi = FALSE, filters = filters, index = 1)
  if (!length(path) || !nzchar(path))
    path <- ask(paste("Paste the full path for", label))

  path <- normalizePath(path, winslash = "/", mustWork = TRUE)
  ext  <- tolower(tools::file_ext(path))
  if (!ext %in% exts)
    stop("Wrong file type for ", label, ". Expected: ", paste0(".", exts, collapse = ", "))
  path
}

pick_boundary <- function() {
  use_gdb <- tolower(ask(
    "Outer boundary stored as a regular file or in a geodatabase? Enter 'file' or 'gdb'", "file"
  )) == "gdb"

  if (!use_gdb)
    return(list(source_type = "file",
                path  = pick_file("Outer area-of-interest boundary", c("gpkg", "shp", "geojson")),
                layer = NA_character_))

  gdb_path <- tryCatch(
    utils::choose.dir(caption = "Select the .gdb folder containing the outer boundary"),
    error = function(e) NA_character_
  )
  if (is.na(gdb_path) || !nzchar(gdb_path))
    gdb_path <- ask("Paste the path to the .gdb folder")

  gdb_path <- normalizePath(gdb_path, winslash = "/", mustWork = TRUE)
  if (tolower(tools::file_ext(gdb_path)) != "gdb")
    stop("Select the folder ending in .gdb, not a file inside it.")

  layers <- sf::st_layers(gdb_path)
  if (!length(layers$name)) stop("No readable spatial layers found in the geodatabase.")

  cat("\nAvailable layers:\n")
  for (i in seq_along(layers$name)) {
    geom <- if (!is.null(layers$geomtype) && length(layers$geomtype) >= i)
              layers$geomtype[i] else "unknown geometry"
    cat(sprintf("  [%d] %s — %s\n", i, layers$name[i], geom))
  }
  flush.console()

  idx <- suppressWarnings(as.integer(ask("Enter the layer number")))
  if (is.na(idx) || idx < 1 || idx > length(layers$name))
    stop("Invalid layer selection.")

  list(source_type = "gdb", path = gdb_path, layer = layers$name[idx])
}

read_boundary <- function(src) {
  sf::st_read(dsn = src$path,
              layer = if (src$source_type == "gdb") src$layer else NULL,
              quiet = TRUE)
}


# ============================================================
# Collect inputs
# ============================================================

lidar_path    <- pick_file("UAV LiDAR point cloud", c("las", "laz"))
chm_path      <- pick_file("Canopy Height Model", c("tif", "tiff"))
workbook_path <- pick_file("Tree inventory workbook (Excel)", "xlsx")
gedi_path     <- pick_file("Clipped GEDI footprints", c("gpkg", "shp", "geojson"))
boundary_src  <- pick_boundary()

aoi <- read_boundary(boundary_src) |> sf::st_make_valid()
if (is.na(sf::st_crs(aoi)))
  stop("The outer boundary has no coordinate reference system.")

output_dir <- file.path(dirname(workbook_path), "uav_biomass_outputs")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)


# ============================================================
# Analysis settings
# ============================================================

cfg <- list(
  expected_footprints         = 6L,
  plot_radius_m               = 12.5,
  plot_diameter_m             = 25,
  mapping_cell_m              = 22.16,   # ~ same area as 25 m-diameter circle
  remove_noise                = TRUE,
  dtm_resolution_m            = 0.4,
  min_tree_height_m           = 2.0,
  treetop_window_m            = 3.0,
  dalponte_th_tree            = 2.0,
  dalponte_th_seed            = 0.45,
  dalponte_th_cr              = 0.55,
  dalponte_max_cr_m           = 10,
  match_max_distance_m        = 4.0,
  match_max_height_difference_m = 8.0,
  match_height_weight         = 0.35
)


# ============================================================
# Confirm selections
# ============================================================

boundary_desc <- if (boundary_src$source_type == "gdb") {
  paste0(boundary_src$path, " | layer: ", boundary_src$layer)
} else {
  boundary_src$path
}

tibble::tibble(
  input = c("UAV LiDAR", "Canopy Height Model", "Tree inventory workbook",
            "GEDI footprints", "Outer boundary", "Output directory"),
  path  = c(lidar_path, chm_path, workbook_path, gedi_path, boundary_desc, output_dir)
)
cfg

## 3. Analysis settings

The defaults are starting values, not universal segmentation parameters. Tune the local-maximum window and Dalponte thresholds against the mapped trees. The paper used Dalponte for a relatively flat managed forest and PTrees for mountainous forest.


In [ ]:
cfg <- list(
  expected_footprints = 6L,
  plot_diameter_m = 25,
  lidar_is_normalized = tolower(readline("Is LAS Z already height above ground? y/n: ")) %in% c("y", "yes"),
  remove_noise = TRUE,
  dtm_resolution_m = 0.4,
  min_tree_height_m = 2.0,
  treetop_window_m = 3.0,
  dalponte_th_tree = 2.0,
  dalponte_th_seed = 0.45,
  dalponte_th_cr = 0.55,
  dalponte_max_cr_m = 10,
  match_max_distance_m = 4.0,
  match_max_height_difference_m = 8.0,
  match_height_weight = 0.35,
  mapping_cell_m = 22.16
)
cfg


## 4. Read and standardize the Excel inventory

The manual/hypsometer fields are treated as reference measurements. Equivalent DBH and biomass are read from `Biomass Calcs`. Manual mapped coordinates (`Lat`, `Long`) are preferred; Working Trees coordinates are the fallback. Multi-stem records are collapsed to one tree record using the workbook's species/tree/footprint identifiers.


In [ ]:
needed_sheets <- c("Footprint Trees", "Biomass Calcs")
if (!all(needed_sheets %in% excel_sheets(workbook_path))) stop("Workbook must contain: ", paste(needed_sheets, collapse = ", "))
stem_raw <- read_excel(workbook_path, sheet = "Footprint Trees")
biomass_raw <- read_excel(workbook_path, sheet = "Biomass Calcs")

first_finite <- function(x) {
  x <- suppressWarnings(as.numeric(x))
  x <- x[is.finite(x)]
  if (length(x)) x[1] else NA_real_
}
as_id <- function(x) {
  y <- suppressWarnings(as.numeric(x))
  ifelse(is.finite(y), format(y, scientific = FALSE, trim = TRUE), as.character(x))
}

# Records to exclude by species.tree.stem identity
exclude_tree_ids <- c(
  "11.1.1",  # footprint 1
  "16.4.1",  # footprint 2 - Pygmy Date Palm, no DBH
  "5.8.1",   # footprint 2 - redwood
  "20.1.1",  # footprint 3
  "24.1.2"   # footprint 5
)

stem_trees <- stem_raw |>
  filter(!is.na(Species), !is.na(`Tree #`)) |>
  mutate(species_id = as_id(Species), tree_no = as_id(`Tree #`), footprint_id = as_id(Footprint)) |>
  group_by(species_id, tree_no, footprint_id) |>
  summarise(
    n_stems_recorded = n_distinct(`Stem #`),
    manual_lat = first_finite(Lat), manual_lon = first_finite(Long),
    wt_lat = first_finite(WT_latitude), wt_lon = first_finite(WT_longitude),
    common_name = dplyr::first(na.omit(`Species - Common Name`), default = NA_character_),
    .groups = "drop"
  ) |>
  mutate(
    latitude = coalesce(manual_lat, wt_lat),
    longitude = coalesce(manual_lon, wt_lon),
    coordinate_source = case_when(
      is.finite(manual_lat) & is.finite(manual_lon) ~ "manual_mapped",
      is.finite(wt_lat) & is.finite(wt_lon) ~ "working_trees",
      TRUE ~ "missing"
    )
  )

tree_reference <- biomass_raw |>
  filter(!is.na(`Spec.`), !is.na(Tree), !is.na(Footprint)) |>
  transmute(
    species_id = as_id(`Spec.`),
    tree_no = as_id(Tree),
    stem_no = as_id(Stem),
    footprint_id = as_id(Footprint),
    tree_id = paste(species_id, tree_no, stem_no, sep = "."),
    scientific_name = `Sci. Species Name`,
    equivalent_dbh_manual_m = suppressWarnings(as.numeric(`GT DBH (m)`)),
    height_manual_m = suppressWarnings(as.numeric(`GT Height`)),
    equivalent_dbh_wt_m = suppressWarnings(as.numeric(`WT DBH (m)`)),
    height_wt_m = suppressWarnings(as.numeric(`WT Height`)),
    biomass_manual_kg = suppressWarnings(as.numeric(`Manual Biomass`)),
    biomass_wt_kg = suppressWarnings(as.numeric(`WT Biomass`)),
    allometry_source = `Allo. Eqn. Source / Density`,
    allometry_code = `Allo Eqn. Code / Type`,
    allometry_equation = `Allometric Equation`,
    wood_density_kg_m3 = suppressWarnings(as.numeric(`DW Density`)),
    biomass_note = Notes
  ) |>
  filter(!tree_id %in% exclude_tree_ids) |>
  left_join(stem_trees, by = c("species_id", "tree_no", "footprint_id")) |>
  mutate(
    has_height_reference = is.finite(height_manual_m) & height_manual_m > 0,
    has_biomass_reference = is.finite(biomass_manual_kg) & biomass_manual_kg > 0,
    has_coordinates = is.finite(latitude) & is.finite(longitude)
  )

# QC summary
complete_footprints <- c("1", "2", "3", "5")

reference_completeness <- tree_reference |>
  filter(footprint_id %in% complete_footprints) |>
  group_by(footprint_id) |>
  summarise(
    n_field_trees = n(),
    n_with_biomass = sum(has_biomass_reference),
    n_missing_biomass = sum(!has_biomass_reference),
    total_biomass_kg = sum(biomass_manual_kg, na.rm = TRUE),
    reference_total_complete = all(has_biomass_reference),
    .groups = "drop"
  )

reference_completeness

inventory_qc <- tree_reference |>
  summarise(
    n_trees = n(),
    n_with_coordinates = sum(has_coordinates),
    n_with_manual_height = sum(has_height_reference),
    n_with_manual_biomass = sum(has_biomass_reference)
  )
inventory_qc

## 5. Load the six footprints and construct mapped tree points

The footprint layer must contain six polygons. Select its footprint-ID column when prompted. IDs should correspond to the workbook's `Footprint` values 1–6. Nonstandard memberships such as `4_6` are flagged and excluded from single-footprint totals unless resolved by the user.


In [ ]:
footprints_path <- gedi_path
footprints <- st_read(footprints_path, quiet = TRUE) |> st_make_valid()
if (nrow(footprints) != cfg$expected_footprints) stop("Expected exactly six footprint polygons; found ", nrow(footprints), ".")
if (is.na(st_crs(footprints))) stop("Footprint layer has no CRS.")
if (st_is_longlat(footprints)) {
  utm_zone <- floor((st_bbox(footprints)[["xmin"]] + 180) / 6) + 1
  hemisphere <- if (st_bbox(footprints)[["ymin"]] >= 0) "north" else "south"
  utm_crs <- st_crs(paste0("+proj=utm +zone=", utm_zone, " +", hemisphere, " +datum=WGS84 +units=m"))
  message("Footprints are in geographic CRS; reprojecting to UTM zone ", utm_zone, " (", hemisphere, ").")
  footprints <- st_transform(footprints, utm_crs)
}

if (!"reading_order_id" %in% names(footprints)) stop("Expected a 'reading_order_id' column in the footprints layer but didn't find one.")
footprints$footprint_id <- as_id(footprints$reading_order_id)
if (anyDuplicated(footprints$footprint_id)) stop("Footprint IDs must be unique.")
footprints$area_m2 <- as.numeric(st_area(footprints))
if (anyDuplicated(footprints$footprint_id)) stop("Footprint IDs must be unique.")
footprints$area_m2 <- as.numeric(st_area(footprints))
expected_area <- pi * (cfg$plot_diameter_m / 2)^2
if (any(abs(footprints$area_m2 - expected_area) / expected_area > 0.10)) warning("At least one footprint area differs from a 12.5 m circle by more than 10%.")

if (sum(tree_reference$has_coordinates) < 2) stop("Too few mapped tree coordinates.")
tree_points <- tree_reference |>
  filter(has_coordinates) |>
  st_as_sf(coords = c("longitude", "latitude"), crs = 4326, remove = FALSE) |>
  st_transform(st_crs(footprints))

spatial_membership <- st_join(tree_points, footprints |> select(spatial_footprint_id = footprint_id), join = st_within, left = TRUE)
membership_qc <- spatial_membership |>
  st_drop_geometry() |>
  mutate(agrees = footprint_id == spatial_footprint_id) |>
  count(coordinate_source, agrees, name = "n")
membership_qc

# Zoom to footprint bounding box with a small buffer
bbox_buf <- st_buffer(st_union(footprints), 10)  # 50 m buffer around all footprints
options(repr.plot.width = 18, repr.plot.height = 14)
ggplot() +
  geom_sf(data = footprints, fill = alpha("steelblue", 0.1), colour = "steelblue", linewidth = 0.8) +
  geom_sf(data = spatial_membership, aes(fill = footprint_id), size = 5, alpha = 0.8, shape = 21, color = "black", stroke = 0.5) +
  geom_sf_text(data = st_centroid(footprints), aes(label = footprint_id),
               size = 8, fontface = "bold", colour = "steelblue4") +
  coord_sf(xlim = c(st_bbox(bbox_buf)["xmin"], st_bbox(bbox_buf)["xmax"]),
           ylim = c(st_bbox(bbox_buf)["ymin"], st_bbox(bbox_buf)["ymax"])) +
  scale_fill_brewer(palette = "Set1") +
  labs(title = "Mapped field trees and 12.5 m footprints",
       subtitle = paste0("n = ", nrow(spatial_membership), " trees across ", nrow(footprints), " footprints"),
       fill = "Footprint ID",
       x = NULL, y = NULL) +
  theme_minimal(base_size = 20) +
  theme(legend.position = "right",
        legend.text = element_text(size = 18),
        legend.title = element_text(size = 19),
        plot.title = element_text(size = 24, face = "bold"),
        plot.subtitle = element_text(size = 18),
        axis.text = element_text(size = 14),
        panel.grid.major = element_line(colour = "grey90"))

In [ ]:
tree_reference |>
  filter(footprint_id %in% footprints$footprint_id) |>
  group_by(footprint_id) |>
  summarise(
    n_field_trees = n(),
    n_with_biomass = sum(has_biomass_reference),
    n_missing_biomass = sum(!has_biomass_reference),
    total_known_biomass_kg = sum(biomass_manual_kg, na.rm = TRUE),
    complete = all(has_biomass_reference)
  )


In [ ]:
install.packages("patchwork")
library(patchwork)

## 6. Load CHM and preprocess LiDAR

The provided CHM is used for treetop detection and Dalponte segmentation. If LAS Z is not normalized, this cell applies SOR noise classification, CSF ground classification, a 0.4 m DTM, and height normalization.


In [ ]:
library(tidyterra)

options(repr.plot.width = 36, repr.plot.height = 12)

p1 <- ggplot() +
  geom_spatraster(data = chm) +
  geom_sf(data = st_transform(footprints, crs(chm) |> st_crs()), 
          fill = NA, colour = "blue", linewidth = 1.5) +
  scale_fill_gradientn(colours = c("darkgreen", "yellow", "white"),
                       na.value = "transparent",
                       name = "Height (m)") +
  labs(title = "Raw CHM") +
  theme_minimal(base_size = 22) +
  theme(plot.title = element_text(size = 28, face = "bold"),
        legend.text = element_text(size = 18),
        legend.title = element_text(size = 20),
        legend.key.height = unit(2, "cm"))

p2 <- ggplot() +
  geom_spatraster(data = chm_smooth) +
  geom_sf(data = st_transform(footprints, crs(chm_smooth) |> st_crs()),
          fill = NA, colour = "blue", linewidth = 1.5) +
  scale_fill_gradientn(colours = c("darkgreen", "yellow", "white"),
                       na.value = "transparent",
                       name = "Height (m)") +
  labs(title = "Smoothed CHM (3x3 median)") +
  theme_minimal(base_size = 22) +
  theme(plot.title = element_text(size = 28, face = "bold"),
        legend.text = element_text(size = 18),
        legend.title = element_text(size = 20),
        legend.key.height = unit(2, "cm"))

p1 + p2

In [ ]:
# Check that all 6 footprints are within the cropped CHM extent
ext(chm)
st_bbox(footprints_chm)

## 7. Segment trees and calculate H, crown diameter, and LBI

LBI follows the paper's 0.5 m vertical slices above 1 m. Slice area is the area of the Delaunay/convex envelope of points in each slice; LBI is the height-weighted sum of slice areas. Inspect crown geometry before using any biomass results.


In [ ]:
# ── Tree detection and crown segmentation ────────────────────────────────────
# This cell detects individual treetops from the smoothed CHM, segments the
# point cloud into individual tree crowns, and calculates crown metrics.
# We use the Dalponte (2016) algorithm as the paper did for their relatively
# flat managed forest site — which is analogous to the flat arboretum terrain.

# ── 1. Load normalized point cloud ──────────────────────────────────────────
# The normalized point cloud has Z = height above ground (not sea level).
# Load raw point cloud using path already selected in cell 2
las <- readLAS(lidar_path,
               select = "xyzrci",
               filter = "-drop_z_below 0")
if (is.empty(las)) stop("LAS is empty or unreadable.")

# Clip to footprints before normalization to reduce memory usage —
# no point normalizing the entire 3.6GB file when we only need the 6 footprint areas
footprints_las <- st_transform(footprints, st_crs(las))
las <- clip_roi(las, footprints_las)
cat("Points after clipping to footprints:", format(npoints(las), big.mark = ","), "\n")

# Classify ground points using Cloth Simulation Filter
las_ground <- classify_ground(las, csf(
  sloop_smooth    = FALSE,
  class_threshold = 0.5,
  cloth_resolution = 0.5,
  rigidness       = 2,
  iterations      = 500,
  time_step       = 0.65
))

# Normalize height: subtract terrain elevation so Z = height above ground
las_norm <- normalize_height(las_ground, tin())
las_norm <- filter_poi(las_norm, Z >= -0.5)
cat("Points after normalization:", format(npoints(las_norm), big.mark = ","), "\n")

# ── 2. Noise removal ─────────────────────────────────────────────────────────
# Statistical Outlier Removal (SOR): flags points that are unusually far from
# their k=8 nearest neighbors by more than m=6 standard deviations.
# These are likely noise rather than real vegetation returns.
if (cfg$remove_noise) {
  las_norm <- classify_noise(las_norm, sor(k = 8, m = 6))
  las_norm <- filter_poi(las_norm, Classification != 18L)
}
cat("Points after noise removal:", format(npoints(las_norm), big.mark = ","), "\n")

# ── 3. Detect treetops ───────────────────────────────────────────────────────
# The local maximum filter (lmf) scans the smoothed CHM and finds pixels that
# are higher than all their neighbors within a search window of treetop_window_m.
# hmin excludes anything below min_tree_height_m from being called a treetop —
# this prevents shrubs and ground clutter from being detected as trees.
# NOTE: at 1m resolution this window covers a 3x3 pixel area. At 0.4m it will
# cover a finer area and may detect more treetops — parameters may need tuning.
ttops <- locate_trees(chm_smooth, 
                      lmf(ws = cfg$treetop_window_m,    # 3m search window
                          hmin = cfg$min_tree_height_m)) # ignore anything < 2m
cat("Treetops detected:", nrow(ttops), "\n")

# ── 4. Dalponte crown segmentation ───────────────────────────────────────────
# Starting from each detected treetop, Dalponte region-growing expands outward
# across the CHM, assigning pixels to the nearest treetop as long as:
# - the pixel height is above th_tree (2m) — avoids growing into gaps/ground
# - the pixel height is above th_seed × treetop height — stays near crown top
# - the pixel height is above th_cr × current region maximum — stops at crown edge
# - the crown radius doesn't exceed max_cr_m (10m)
# The arboretum has a mix of species and complex structure — if segmentation
# looks poor after inspection, th_seed and th_cr are the main parameters to tune.
seg_algorithm <- dalponte2016(
  chm_smooth, ttops,
  th_tree = cfg$dalponte_th_tree,    # 2.0 m minimum tree height
  th_seed = cfg$dalponte_th_seed,    # 0.45 — seed pixel must be 45% of treetop height
  th_cr   = cfg$dalponte_th_cr,      # 0.55 — crown pixel must be 55% of region max
  max_cr  = cfg$dalponte_max_cr_m    # 10 m maximum crown radius
)

# Apply segmentation to the normalized point cloud —
# each point gets assigned a treeID based on which crown it falls under
las_segmented <- segment_trees(las_norm, seg_algorithm)

# ── 5. Calculate crown metrics ───────────────────────────────────────────────
# For each segmented tree, extract:
# - lidar_height_m: the maximum Z value (tallest point in the crown)
# - crown_area_m2: area of the convex hull around all crown points
# - crown_diameter_m: equivalent circular diameter from crown area
# These match the paper's H and CD metrics used in the tree-centric ASM model.
crowns <- crown_metrics(las_segmented, 
                        ~list(lidar_height_m = max(Z, na.rm = TRUE)), 
                        geom = "convex") |>
  st_make_valid() |>
  mutate(
    crown_area_m2    = as.numeric(st_area(geometry)),
    crown_diameter_m = 2 * sqrt(crown_area_m2 / pi)
  )

# ── 6. Calculate LBI (LiDAR Biomass Index) ───────────────────────────────────
# LBI integrates crown area across height slices, weighting by height.
# It captures the 3D volume of the crown rather than just its 2D footprint.
# Following the paper: slices start at 1m (avoids shrub interference),
# slice thickness dz = 0.5m, area per slice = convex hull of points in that slice.
# LBI = sum(slice_area × slice_height × dz) across all slices
slice_area <- function(dat) {
  if (nrow(dat) < 3) return(0)
  xy <- unique(dat[, c("X", "Y")])
  if (nrow(xy) < 3) return(0)
  poly <- st_as_sf(xy, coords = c("X", "Y"), crs = st_crs(las_norm)) |>
    st_union() |> st_convex_hull()
  as.numeric(st_area(poly))
}

compute_lbi <- function(dat, bottom_m = 1, dz = 0.5) {
  top <- max(dat$Z, na.rm = TRUE)
  if (!is.finite(top) || top <= bottom_m) return(NA_real_)
  lower <- seq(bottom_m, top, by = dz)
  areas <- map_dbl(lower, function(h) 
    slice_area(dat[dat$Z >= h & dat$Z < h + dz, , drop = FALSE]))
  sum(areas * lower * dz, na.rm = TRUE)
}

segment_data <- as_tibble(las_segmented@data) |>
  filter(!is.na(treeID)) |>
  group_split(treeID)
lbi_table <- map_dfr(segment_data, 
                     ~tibble(treeID = first(.x$treeID), 
                             LBI    = compute_lbi(.x)))
crowns <- crowns |> left_join(lbi_table, by = "treeID")

cat("Segmented crowns:", nrow(crowns), 
    "| Field trees with coordinates:", nrow(tree_points), "\n")

# ── 7. QA plot ───────────────────────────────────────────────────────────────
# Red crosses = field-mapped tree locations
# Grey polygons = LiDAR-segmented crowns
# Ideally each red cross should fall inside a grey polygon.
# Over-segmentation = too many small polygons per tree
# Under-segmentation = one large polygon covering multiple trees
options(repr.plot.width = 20, repr.plot.height = 14)
ggplot() +
  geom_sf(data = st_transform(crowns, st_crs(footprints)), 
          fill = NA, colour = "grey40", linewidth = 0.5) +
  geom_sf(data = footprints, fill = NA, colour = "blue", linewidth = 1) +
  geom_sf(data = tree_points, colour = "red", size = 2, shape = 3) +
  labs(title = "Segmentation QA: crowns and mapped trees",
       subtitle = paste0(nrow(crowns), " crowns detected across ", 
                         nrow(footprints), " footprints")) +
  theme_minimal(base_size = 18)

## 8. One-to-one field/LiDAR matching

The Hungarian assignment prevents several field trees from matching the same crown. Cost combines horizontal distance and height disagreement. Matches beyond either threshold are rejected. Review the map and exported match table manually; this is still not equivalent to the paper's manual crown-shape matching.


In [ ]:
crown_points <- st_point_on_surface(crowns)
field_for_match <- tree_points |> filter(has_height_reference)
dist_m <- units::drop_units(st_distance(field_for_match, crown_points))
height_diff <- abs(outer(field_for_match$height_manual_m, crowns$lidar_height_m, "-"))
valid_pair <- dist_m <= cfg$match_max_distance_m & height_diff <= cfg$match_max_height_difference_m
cost <- dist_m / cfg$match_max_distance_m + cfg$match_height_weight * height_diff / cfg$match_max_height_difference_m
cost[!valid_pair] <- 1e6

n_square <- max(nrow(cost), ncol(cost))
padded_cost <- matrix(2, n_square, n_square)
padded_cost[seq_len(nrow(cost)), seq_len(ncol(cost))] <- cost
assignment <- as.integer(solve_LSAP(padded_cost))
matched_rows <- tibble(field_row = seq_len(nrow(cost)), crown_row = assignment[seq_len(nrow(cost))]) |>
  filter(crown_row <= ncol(cost)) |>
  mutate(match_valid = map2_lgl(field_row, crown_row, ~valid_pair[.x, .y])) |>
  filter(match_valid)

matches <- matched_rows |>
  transmute(
    tree_id = field_for_match$tree_id[field_row],
    treeID = crowns$treeID[crown_row],
    match_distance_m = dist_m[cbind(field_row, crown_row)],
    height_difference_m = crowns$lidar_height_m[crown_row] - field_for_match$height_manual_m[field_row]
  ) |>
  left_join(st_drop_geometry(tree_reference), by = "tree_id") |>
  left_join(st_drop_geometry(crowns) |> select(treeID, lidar_height_m, crown_area_m2, crown_diameter_m, LBI), by = "treeID")

matching_qc <- tibble(
  field_trees_with_height = nrow(field_for_match), matched = nrow(matches),
  match_rate_pct = 100 * nrow(matches) / nrow(field_for_match),
  height_bias_m = mean(matches$height_difference_m),
  height_rmse_m = sqrt(mean(matches$height_difference_m^2))
)
matching_qc
write_csv(matches, file.path(output_dir, "field_lidar_tree_matches.csv"))


In [ ]:
ggplot(matches, aes(x = height_manual_m, y = lidar_height_m, colour = match_distance_m)) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) + geom_point(size = 2) +
  scale_colour_viridis_c() + coord_equal() +
  labs(x = "Manual/hypsometer height (m)", y = "UAV–LiDAR height (m)", colour = "Match distance (m)",
       title = "Height validation for one-to-one matches") + theme_minimal()


## 9. Tree-centric biomass models with footprint-level cross-validation

Two paper-aligned models are fitted only to matched trees with manual biomass:

- local ASM: `ln(Biomass) = α + β ln(H × CD)`
- LBI model: `ln(Biomass) = α + β ln(H) + γ ln(LBI)`

Entire footprints—not individual trees—are left out during cross-validation so trees from the same footprint do not appear in both training and testing sets. A smearing correction is applied after log back-transformation.


In [ ]:
calibration <- matches |>
  filter(has_biomass_reference, biomass_manual_kg > 0, lidar_height_m > 0, crown_diameter_m > 0, LBI > 0,
         footprint_id %in% footprints$footprint_id)
if (nrow(calibration) < 15) stop("Fewer than 15 valid matched biomass trees remain; revise segmentation/matching before modeling.")

fit_log_model <- function(formula, dat) {
  fit <- lm(formula, data = dat)
  list(fit = fit, smear = mean(exp(residuals(fit))))
}
predict_log_model <- function(object, newdata) exp(predict(object$fit, newdata = newdata)) * object$smear

asm_formula <- log(biomass_manual_kg) ~ log(lidar_height_m * crown_diameter_m)
lbi_formula <- log(biomass_manual_kg) ~ log(lidar_height_m) + log(LBI)
asm_model <- fit_log_model(asm_formula, calibration)
lbi_model <- fit_log_model(lbi_formula, calibration)
print(summary(asm_model$fit))
print(summary(lbi_model$fit))

lopo_predict <- function(formula, dat, model_name) {
  map_dfr(unique(dat$footprint_id), function(fp) {
    train <- filter(dat, footprint_id != fp)
    test <- filter(dat, footprint_id == fp)
    model <- fit_log_model(formula, train)
    test |>
      transmute(model = model_name, footprint_id, tree_id,
                observed_kg = biomass_manual_kg, predicted_kg = predict_log_model(model, test))
  })
}
tree_cv <- bind_rows(
  lopo_predict(asm_formula, calibration, "Local ASM"),
  lopo_predict(lbi_formula, calibration, "LBI")
) |>
  mutate(error_kg = predicted_kg - observed_kg)
tree_cv_accuracy <- tree_cv |>
  group_by(model) |>
  summarise(n = n(), bias_kg = mean(error_kg), rmse_kg = sqrt(mean(error_kg^2)),
            rrmse_pct = 100 * rmse_kg / mean(observed_kg),
            pseudo_r2 = 1 - sum(error_kg^2) / sum((observed_kg - mean(observed_kg))^2), .groups = "drop")
tree_cv_accuracy
write_csv(tree_cv, file.path(output_dir, "tree_level_footprint_cv_predictions.csv"))
write_csv(tree_cv_accuracy, file.path(output_dir, "tree_level_cv_accuracy.csv"))


## 10. Predict all segmented trees and aggregate to footprints

This produces a LiDAR estimate for every segmented crown, including trees whose field DBH/biomass was missing. It does not invent a field biomass value for those trees.


In [ ]:
crown_predictions <- crowns |>
  filter(lidar_height_m > 0, crown_diameter_m > 0, LBI > 0)
prediction_data <- st_drop_geometry(crown_predictions)
crown_predictions$pred_asm_kg <- predict_log_model(asm_model, prediction_data)
crown_predictions$pred_lbi_kg <- predict_log_model(lbi_model, prediction_data)
crown_locations <- st_point_on_surface(crown_predictions) |>
  st_join(footprints |> select(footprint_id, area_m2), join = st_within, left = FALSE)

lidar_footprint_biomass <- crown_locations |>
  st_drop_geometry() |>
  group_by(footprint_id, area_m2) |>
  summarise(n_segmented_trees = n(), asm_total_kg = sum(pred_asm_kg), lbi_total_kg = sum(pred_lbi_kg), .groups = "drop") |>
  mutate(asm_Mg_ha = asm_total_kg / area_m2 * 10, lbi_Mg_ha = lbi_total_kg / area_m2 * 10)

reference_completeness <- tree_reference |>
  filter(footprint_id %in% footprints$footprint_id) |>
  group_by(footprint_id) |>
  summarise(n_field_trees = n(), n_with_reference_biomass = sum(has_biomass_reference),
            n_missing_reference_biomass = sum(!has_biomass_reference),
            observed_partial_biomass_kg = sum(biomass_manual_kg, na.rm = TRUE),
            reference_total_complete = all(has_biomass_reference), .groups = "drop")

footprint_results <- lidar_footprint_biomass |>
  left_join(reference_completeness, by = "footprint_id")
footprint_results
write_csv(st_drop_geometry(crown_predictions), file.path(output_dir, "segmented_tree_biomass_predictions.csv"))
write_csv(footprint_results, file.path(output_dir, "footprint_biomass_predictions_and_reference_completeness.csv"))
st_write(crown_predictions, file.path(output_dir, "segmented_tree_biomass.gpkg"), delete_dsn = TRUE, quiet = TRUE)


## 11. Extract paper-aligned footprint metrics

These CHM-, point-, and voxel-based metrics are saved even though an area model is not fitted. They support QA now and make the notebook reusable after more complete independent plots are collected. `lai_proxy` is clearly labeled because the paper did not provide enough implementation detail to reproduce its LAI calculation exactly.


In [ ]:
entropy_1m <- function(z) {
  z <- z[is.finite(z) & z >= 1]
  if (length(z) < 2) return(NA_real_)
  breaks <- seq(floor(min(z)), ceiling(max(z)) + 1, by = 1)
  p <- hist(z, breaks = breaks, plot = FALSE)$counts
  p <- p[p > 0] / sum(p)
  -sum(p * log(p))
}

footprint_metric <- function(cloud, polygon, id) {
  if (is.null(cloud) || is.empty(cloud) || npoints(cloud) < 20) return(tibble(footprint_id = id, extraction_failed = TRUE))
  z <- cloud@data$Z
  chm_clip <- crop(chm, vect(st_transform(polygon, crs(chm))), mask = TRUE)
  cv <- values(chm_clip, mat = FALSE)
  cv <- cv[is.finite(cv)]
  pts <- as_tibble(cloud@data) |>
    filter(is.finite(Z), Z >= 1) |>
    mutate(gx = floor(X), gy = floor(Y)) |>
    group_by(gx, gy) |>
    summarise(hp10 = quantile(Z, 0.10, names = FALSE), hp75 = quantile(Z, 0.75, names = FALSE), .groups = "drop")
  gap <- mean(z < 1, na.rm = TRUE)
  tibble(
    footprint_id = id, extraction_failed = FALSE, n_points = length(z),
    point_density_m2 = length(z) / as.numeric(st_area(polygon)),
    mu_chm = mean(cv, na.rm = TRUE), sigma_chm = sd(cv, na.rm = TRUE), gf_chm_below3 = mean(cv < 3, na.rm = TRUE),
    density_3 = mean(z > 3, na.rm = TRUE), density_6 = mean(z > 6, na.rm = TRUE),
    density_9 = mean(z > 9, na.rm = TRUE), density_12 = mean(z > 12, na.rm = TRUE),
    density_15 = mean(z > 15, na.rm = TRUE), density_18 = mean(z > 18, na.rm = TRUE),
    fhd = entropy_1m(z), lai_proxy = ifelse(gap > 0 & gap < 1, -log(gap) / 0.5, NA_real_),
    mu_hp10 = mean(pts$hp10, na.rm = TRUE), mu_hp75 = mean(pts$hp75, na.rm = TRUE),
    sigma_hp75 = sd(pts$hp75, na.rm = TRUE)
  )
}

footprint_clouds <- clip_roi(las_norm, footprints)
if (!is.list(footprint_clouds)) footprint_clouds <- list(footprint_clouds)
footprint_lidar_metrics <- map2_dfr(footprint_clouds, seq_len(nrow(footprints)),
  ~footprint_metric(.x, footprints[.y, ], footprints$footprint_id[.y]))
if (max(footprint_lidar_metrics$point_density_m2, na.rm = TRUE) / min(footprint_lidar_metrics$point_density_m2, na.rm = TRUE) > 1.5) {
  warning("Point density varies by more than 50% among footprints; density standardization should be investigated.")
}
write_csv(footprint_lidar_metrics, file.path(output_dir, "footprint_lidar_metrics.csv"))
footprint_lidar_metrics


## 12. Area-based biomass estimation

The area-based approach links plot-level biomass to LiDAR-derived structural metrics 
extracted from the point cloud and CHM, without requiring individual tree segmentation.
This follows the paper's area-based methodology (Section 3.2.3 and 3.3.3), which used
a multiplicative power (MP) model of the form:

ln(Biomass) = α + α₁ln(P₁) + α₂ln(P₂) + ... + αₙln(Pₙ)

Because we only have 4 complete reference footprints (vs the paper's 23-52 plots),
we fit a single-predictor version of this model. Leave-one-out cross-validation (LOOCV)
is used to select the best predictor and assess accuracy — matching the paper's 
validation approach. Footprints 4 and 6 are predicted only, with no reference biomass
for validation.

In [ ]:
# ── Area-based biomass model ────────────────────────────────────────────────

# Join the LiDAR metrics extracted in cell 11 with the reference biomass totals
# from cell 4. Only keep the 4 footprints with complete field biomass measurements
# for model calibration — footprints 4 and 6 will be predicted only.
area_calibration <- footprint_lidar_metrics |>
  filter(!extraction_failed) |>
  left_join(reference_completeness, by = "footprint_id") |>
  filter(reference_total_complete) |>
  # Log-transform biomass for the multiplicative power model (paper Equation 9)
  mutate(log_biomass = log(total_biomass_kg))

# These are the candidate predictors from the paper's Table 1:
# CHM-based: mean and SD of canopy height, gap fraction below 3m
# Point-based: proportion of returns above height thresholds, foliage height diversity, LAI
# Voxel-based: mean and SD of height percentiles within voxels
candidate_predictors <- c(
  "mu_chm", "sigma_chm", "gf_chm_below3",
  "density_3", "density_6", "density_9", "density_12", "density_15", "density_18",
  "fhd", "lai_proxy",
  "mu_hp10", "mu_hp75", "sigma_hp75"
)

# ── LOOCV for each single predictor ────────────────────────────────────────
# For each candidate metric, we fit a log-log model leaving one footprint out
# at a time, predict that footprint's biomass, then compare to observed.
# This mirrors the paper's LOOCV approach but with 4 plots instead of 23-52.
# The smearing correction (Duan 1983) adjusts for bias introduced by
# back-transforming from log scale to kg.
loocv_single <- map_dfr(candidate_predictors, function(pred) {
  dat <- area_calibration |>
    select(footprint_id, log_biomass, x = all_of(pred)) |>
    filter(is.finite(x), is.finite(log_biomass))
  
  # Skip predictors with fewer than 3 valid footprints
  if (nrow(dat) < 3) return(tibble(predictor = pred, n = nrow(dat), r2 = NA, rmse_kg = NA, rrmse_pct = NA))
  
  # Leave one footprint out, train on the rest, predict the held-out footprint
  preds <- map_dbl(seq_len(nrow(dat)), function(i) {
    train <- dat[-i, ]
    test  <- dat[i, ]
    fit   <- lm(log_biomass ~ log(abs(x) + 1e-6), data = train)
    # Smearing correction: accounts for Jensen's inequality when back-transforming
    smear <- mean(exp(residuals(fit)))
    exp(predict(fit, newdata = test)) * smear
  })
  
  # Calculate accuracy metrics: R², RMSE, and relative RMSE (paper's Equations 10-12)
  observed <- exp(dat$log_biomass)
  errors   <- preds - observed
  tibble(
    predictor  = pred,
    n          = nrow(dat),
    r2         = 1 - sum(errors^2) / sum((observed - mean(observed))^2),
    rmse_kg    = sqrt(mean(errors^2)),
    rrmse_pct  = 100 * sqrt(mean(errors^2)) / mean(observed)
  )
}) |>
  # Sort by rRMSE so the best predictor is at the top
  arrange(rrmse_pct)

loocv_single

# ── Fit final model using the best predictor ────────────────────────────────
# The best predictor is whichever had the lowest rRMSE across LOOCV folds.
# We refit on all 4 complete footprints for the final prediction model.
best_pred <- loocv_single$predictor[1]
cat("Best predictor:", best_pred, "\n")

area_dat <- area_calibration |>
  select(footprint_id, log_biomass, x = all_of(best_pred)) |>
  filter(is.finite(x), is.finite(log_biomass))

area_model <- lm(log_biomass ~ log(abs(x) + 1e-6), data = area_dat)
area_smear <- mean(exp(residuals(area_model)))

# ── Predict biomass for all 6 footprints ───────────────────────────────────
# Apply the fitted model to all 6 footprints, including 4 and 6 which have
# no complete reference biomass. Those are flagged as prediction-only.
area_predictions <- footprint_lidar_metrics |>
  filter(!extraction_failed) |>
  left_join(reference_completeness |> select(footprint_id, total_biomass_kg, reference_total_complete),
            by = "footprint_id") |>
  mutate(
    x = .data[[best_pred]],
    predicted_biomass_kg = exp(predict(area_model, newdata = data.frame(x = x))) * area_smear,
    # Flag which footprints were used for calibration vs prediction only
    calibration_footprint = replace_na(reference_total_complete, FALSE)
  ) |>
  select(footprint_id, predicted_biomass_kg, observed_biomass_kg = total_biomass_kg, calibration_footprint)

area_predictions

# ── Plot predicted vs observed for the 4 calibration footprints ─────────────
# Points on or near the dashed 1:1 line indicate accurate predictions.
# With only 4 points this is illustrative rather than statistically robust.
ggplot(area_predictions |> filter(calibration_footprint),
       aes(x = observed_biomass_kg, y = predicted_biomass_kg, label = footprint_id)) +
  geom_abline(slope = 1, intercept = 0, linetype = 2, colour = "grey50") +
  geom_point(size = 4, colour = "steelblue") +
  ggrepel::geom_text_repel(size = 5) +
  labs(title = "Area-based model: predicted vs observed biomass",
       subtitle = paste0("Best predictor: ", best_pred, " | LOOCV on 4 footprints"),
       x = "Observed biomass (kg)", y = "Predicted biomass (kg)") +
  theme_minimal(base_size = 16)

write_csv(area_predictions, file.path(output_dir, "area_based_biomass_predictions.csv"))

## 13. Optional AOI prediction and reproducibility record

A wall-to-wall AOI map should only be generated after crown segmentation and held-out footprint performance are acceptable. The same fitted tree model can then be applied to all segmented crowns within the AOI. Cells outside the range of calibration heights, crown diameters, or LBI values should be flagged as extrapolation.


In [ ]:
write_csv(selected_inputs, file.path(output_dir, "selected_inputs.csv"))
write_csv(st_drop_geometry(tree_points), file.path(output_dir, "standardized_mapped_tree_reference.csv"))
writeLines(capture.output(sessionInfo()), file.path(output_dir, "sessionInfo.txt"))
cat("Outputs written to:", normalizePath(output_dir, winslash = "/", mustWork = TRUE), "\n")
sessionInfo()
